# Python Challenge 4 – Real-Time Weather Logger (API + CSV)
Fetches current weather for a city from the **OpenWeatherMap** API and logs it to `weather_log.csv`.

**Columns:** Date (auto-filled with today) · City · Temperature (°C) · Condition

**Features**
1. Ask for a city name
2. Fetch weather from OpenWeatherMap
3. Save to `weather_log.csv`
4. Prevent duplicate entries for the same city on the same day
5. Menu: add new log · view all logs · statistics (average / highest / lowest temperature, most frequent condition)

**Bonus:** table output · graceful handling of API failures and invalid city names.

### Getting an API key (free)
1. Create an account at <https://openweathermap.org/api> and copy your key (new keys can take a little while to activate).
2. Put it in an environment variable named `OPENWEATHER_API_KEY` (or paste it in the cell below).

> The demo further down uses a **simulated API**, so the notebook runs even without internet or a key. The real function is `fetch_weather()`.

In [1]:
import csv
import os
import requests
from collections import Counter
from datetime import date

LOG_FILE = "weather_log.csv"
FIELDS = ["Date", "City", "Temperature_C", "Condition"]

# Paste your key here OR set the OPENWEATHER_API_KEY environment variable
API_KEY = os.environ.get("OPENWEATHER_API_KEY", "")
API_URL = "https://api.openweathermap.org/data/2.5/weather"

## 1. Fetching weather from the API
`units=metric` makes the API return °C directly. Every failure mode is converted into a `WeatherError` with a friendly message:

| Situation | Handling |
|---|---|
| no API key | message telling the user how to set one |
| HTTP 404 | "city not found" |
| HTTP 401 | invalid API key |
| timeout / no internet | connection message |
| unexpected response | generic message |

In [2]:
class WeatherError(Exception):
    """Raised for any problem getting weather data (shown to the user as a friendly message)."""

def fetch_weather(city, api_key=None):
    """Return (temperature_in_celsius, condition) for the city using OpenWeatherMap."""
    api_key = api_key or API_KEY
    if not api_key:
        raise WeatherError("No API key found. Set the OPENWEATHER_API_KEY environment variable.")
    try:
        resp = requests.get(API_URL,
                            params={"q": city, "appid": api_key, "units": "metric"},
                            timeout=10)
    except requests.exceptions.Timeout:
        raise WeatherError("The weather service took too long to respond. Try again.")
    except requests.exceptions.ConnectionError:
        raise WeatherError("Could not connect. Please check your internet connection.")
    except requests.exceptions.RequestException as e:
        raise WeatherError(f"Network problem: {e}")

    if resp.status_code == 404:
        raise WeatherError(f"City '{city}' was not found. Check the spelling.")
    if resp.status_code == 401:
        raise WeatherError("Invalid API key.")
    if resp.status_code != 200:
        raise WeatherError(f"Weather service error (HTTP {resp.status_code}).")

    try:
        data = resp.json()
        return round(float(data["main"]["temp"]), 1), data["weather"][0]["main"]
    except (ValueError, KeyError, IndexError, TypeError):
        raise WeatherError("Received an unexpected response from the weather service.")

## 2. CSV helpers

In [3]:
def ensure_log(path=LOG_FILE):
    if not os.path.exists(path):
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()

def load_logs(path=LOG_FILE):
    ensure_log(path)
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def add_log(city, fetcher=fetch_weather, today=None, path=LOG_FILE):
    """Fetch weather for `city` and append it. Returns (ok, message).
    Duplicate rule: one entry per city per day (city comparison is case-insensitive)."""
    city = city.strip()
    if not city:
        return False, "City name cannot be empty."
    today = (today or date.today()).isoformat()

    # Check for a duplicate *before* calling the API (saves an API request)
    if any(r["City"].lower() == city.lower() and r["Date"] == today for r in load_logs(path)):
        return False, f"{city.title()} has already been logged for {today}."

    try:
        temp, condition = fetcher(city)
    except WeatherError as e:
        return False, f"Could not log weather: {e}"

    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow(
            {"Date": today, "City": city.title(), "Temperature_C": temp, "Condition": condition})
    return True, f"Logged {city.title()}: {temp}°C, {condition} ({today})."

## 3. Viewing logs (table) and statistics

In [4]:
def print_table(rows):
    if not rows:
        print("  (no logs yet)")
        return
    w_city = max(len("City"), *(len(r["City"]) for r in rows))
    w_cond = max(len("Condition"), *(len(r["Condition"]) for r in rows))
    line = f"+------------+-{'-' * w_city}-+-----------+-{'-' * w_cond}-+"
    print(line)
    print(f"| {'Date':<10} | {'City':<{w_city}} | {'Temp (°C)':>9} | {'Condition':<{w_cond}} |")
    print(line)
    for r in rows:
        print(f"| {r['Date']:<10} | {r['City']:<{w_city}} | {float(r['Temperature_C']):>9.1f} | {r['Condition']:<{w_cond}} |")
    print(line)

def show_stats(path=LOG_FILE):
    rows = load_logs(path)
    if not rows:
        print("No data to analyse yet.")
        return
    temps = [float(r["Temperature_C"]) for r in rows]
    hottest = max(rows, key=lambda r: float(r["Temperature_C"]))
    coldest = min(rows, key=lambda r: float(r["Temperature_C"]))
    condition, count = Counter(r["Condition"] for r in rows).most_common(1)[0]

    print("----- Weather Statistics -----")
    print(f"Records logged      : {len(rows)}")
    print(f"Average temperature : {sum(temps) / len(temps):.1f} °C")
    print(f"Highest temperature : {hottest['Temperature_C']} °C ({hottest['City']}, {hottest['Date']})")
    print(f"Lowest temperature  : {coldest['Temperature_C']} °C ({coldest['City']}, {coldest['Date']})")
    print(f"Most frequent cond. : {condition} ({count} time{'s' if count > 1 else ''})")

## 4. Menu loop

In [5]:
def main(path=LOG_FILE, fetcher=fetch_weather):
    ensure_log(path)
    while True:
        print("\n===== WEATHER LOGGER =====")
        print("1. Add new weather log")
        print("2. View all logs")
        print("3. Show statistics")
        print("4. Exit")
        choice = input("Choose an option (1-4): ").strip()

        if choice == "1":
            city = input("Enter city name: ")
            print(add_log(city, fetcher=fetcher, path=path)[1])
        elif choice == "2":
            print_table(load_logs(path))
        elif choice == "3":
            show_stats(path)
        elif choice == "4":
            print("Goodbye!")
            break
        else:
            print("Invalid option, please choose 1-4.")

## 5. Offline demo with a simulated API
`fake_fetch` imitates OpenWeatherMap: it returns data for known cities and raises the *same* `WeatherError` the real function would for an unknown city or a network failure. This lets us test every path without a key.

In [6]:
FAKE_DATA = {
    "pune":      (27.4, "Clouds"),
    "mumbai":    (31.2, "Clear"),
    "delhi":     (35.8, "Clear"),
    "london":    (12.1, "Rain"),
    "new york":  (18.6, "Clear"),
}

def fake_fetch(city):
    if city.lower() == "offline-town":
        raise WeatherError("Could not connect. Please check your internet connection.")
    if city.lower() not in FAKE_DATA:
        raise WeatherError(f"City '{city}' was not found. Check the spelling.")
    return FAKE_DATA[city.lower()]

if os.path.exists("demo_weather_log.csv"):
    os.remove("demo_weather_log.csv")

### Menu run (scripted input)

In [7]:
from unittest.mock import patch

def run_with_inputs(func, inputs, **kwargs):
    it = iter(inputs)
    def fake_input(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    with patch("builtins.input", fake_input):
        return func(**kwargs)

run_with_inputs(main, [
    "1", "Pune",
    "1", "Mumbai",
    "1", "pune",            # duplicate for today -> blocked
    "1", "Atlantis",        # invalid city -> friendly error
    "1", "offline-town",    # network failure -> friendly error
    "2",
    "3",
    "4",
], path="demo_weather_log.csv", fetcher=fake_fetch)


===== WEATHER LOGGER =====
1. Add new weather log
2. View all logs
3. Show statistics
4. Exit
Choose an option (1-4): 1
Enter city name: Pune
Logged Pune: 27.4°C, Clouds (2026-10-10).

===== WEATHER LOGGER =====
1. Add new weather log
2. View all logs
3. Show statistics
4. Exit
Choose an option (1-4): 1
Enter city name: Mumbai
Logged Mumbai: 31.2°C, Clear (2026-10-10).

===== WEATHER LOGGER =====
1. Add new weather log
2. View all logs
3. Show statistics
4. Exit
Choose an option (1-4): 1
Enter city name: pune
Pune has already been logged for 2026-10-10.

===== WEATHER LOGGER =====
1. Add new weather log
2. View all logs
3. Show statistics
4. Exit
Choose an option (1-4): 1
Enter city name: Atlantis
Could not log weather: City 'Atlantis' was not found. Check the spelling.

===== WEATHER LOGGER =====
1. Add new weather log
2. View all logs
3. Show statistics
4. Exit
Choose an option (1-4): 1
Enter city name: offline-town
Could not log weather: Could not connect. Please check your interne

### Simulating several days
`add_log` accepts a `today` argument so we can back-fill other dates; this shows the statistics over a longer period. The same city *can* be logged again on a different day.

In [8]:
from datetime import timedelta

base = date.today()
add_log("Pune",   fetcher=fake_fetch, today=base - timedelta(days=1), path="demo_weather_log.csv")
add_log("Delhi",  fetcher=fake_fetch, today=base - timedelta(days=1), path="demo_weather_log.csv")
add_log("London", fetcher=fake_fetch, today=base - timedelta(days=2), path="demo_weather_log.csv")
add_log("Mumbai", fetcher=fake_fetch, today=base - timedelta(days=2), path="demo_weather_log.csv")

print_table(sorted(load_logs("demo_weather_log.csv"), key=lambda r: r["Date"]))
print()
show_stats("demo_weather_log.csv")

+------------+--------+-----------+-----------+
| Date       | City   | Temp (°C) | Condition |
+------------+--------+-----------+-----------+
| 2026-10-08 | London |      12.1 | Rain      |
| 2026-10-08 | Mumbai |      31.2 | Clear     |
| 2026-10-09 | Pune   |      27.4 | Clouds    |
| 2026-10-09 | Delhi  |      35.8 | Clear     |
| 2026-10-10 | Pune   |      27.4 | Clouds    |
| 2026-10-10 | Mumbai |      31.2 | Clear     |
+------------+--------+-----------+-----------+

----- Weather Statistics -----
Records logged      : 6
Average temperature : 27.5 °C
Highest temperature : 35.8 °C (Delhi, 2026-10-09)
Lowest temperature  : 12.1 °C (London, 2026-10-08)
Most frequent cond. : Clear (3 times)


### What the real function does without a key
This shows the graceful failure path of the *real* `fetch_weather` (no crash, just a clear message).

In [9]:
try:
    fetch_weather("Pune", api_key="")
except WeatherError as e:
    print("WeatherError:", e)

WeatherError: No API key found. Set the OPENWEATHER_API_KEY environment variable.


In [10]:
os.remove("demo_weather_log.csv")
print("demo_weather_log.csv removed")

demo_weather_log.csv removed


## 6. Using the real API
1. Set `API_KEY` (top of the notebook) or the `OPENWEATHER_API_KEY` environment variable.
2. Run the cell below. It uses the real `fetch_weather` and writes to `weather_log.csv`.

In [11]:
# main()